In [21]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
from nltk.translate.bleu_score import corpus_bleu
import random
import time
import platform
import logging
from tqdm import tqdm
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
import os
#import mlflow

# === Device Selection Option ===
# Set this to 'auto', 'cuda', 'mps', or 'cpu' to force device selection
DEVICE_OPTION = 'auto'  # 'auto', 'cuda', 'mps', or 'cpu'
# Logger setup
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# Device selection logic
if DEVICE_OPTION == 'cuda' and torch.cuda.is_available():
    device = torch.device('cuda')
    logger.info('Using CUDA (GPU)')
elif DEVICE_OPTION == 'mps' and torch.backends.mps.is_available():
    device = torch.device('mps')
    logger.info('Using Apple Silicon MPS (M1/M2/M3/M4)')
elif DEVICE_OPTION == 'cpu':
    device = torch.device('cpu')
    logger.info('Using CPU')
else:
    # Auto mode: prefer CUDA, then MPS, then CPU
    if torch.cuda.is_available():
        device = torch.device('cuda')
        logger.info('Auto-selected CUDA (GPU)')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        logger.info('Auto-selected Apple Silicon MPS (M1/M2/M3/M4)')
    else:
        device = torch.device('cpu')
        logger.info('Auto-selected CPU')
logger.info(f'Final device: {device}')

# Checkpoint directory
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
eng_sample = pd.read_csv('/content/drive/MyDrive/preprocessed_data/eng_preprocessed.csv')
nl_sample = pd.read_csv('/content/drive/MyDrive/preprocessed_data/nl_preprocessed.csv')
## activated only while hyperperameter tuning
sample_indices = eng_sample.sample(frac=0.3 , random_state=42).index

eng_sample = eng_sample.loc[sample_indices].reset_index(drop=True)
nl_sample = nl_sample.loc[sample_indices].reset_index(drop=True)
# 2. Tokenization
def tokenize(sentences):
    return [str(s).split() for s in sentences]

eng_tokens = tokenize(eng_sample['sentence'])
nl_tokens = [['<SOS>'] + str(s).split() + ['<EOS>'] for s in nl_sample['sentence']]

# 3. Build vocabularies
def build_vocab(token_lists, min_freq=2):
    counter = Counter(token for sent in token_lists for token in sent)
    vocab = {'<PAD>':0, '<SOS>':1, '<EOS>':2, '<UNK>':3}
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab

eng_vocab = build_vocab(eng_tokens, min_freq=2)
nl_vocab = build_vocab(nl_tokens, min_freq=2)

def encode(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

eng_indices = [encode(sent, eng_vocab) for sent in eng_tokens]
nl_indices = [encode(sent, nl_vocab) for sent in nl_tokens]

def pad_sequences(sequences, max_len, pad_value=0):
    return [seq + [pad_value]*(max_len - len(seq)) if len(seq) < max_len else seq[:max_len] for seq in sequences]

max_len_eng = min(50, max(len(seq) for seq in eng_indices))
max_len_nl = min(50, max(len(seq) for seq in nl_indices))

eng_padded = pad_sequences(eng_indices, max_len_eng)
nl_padded = pad_sequences(nl_indices, max_len_nl)

# Compute lengths for packing
eng_lengths = [min(len(seq), max_len_eng) for seq in eng_indices]

# 5. Split data
X_train, X_test, y_train, y_test, len_train, len_test = train_test_split(
    eng_padded, nl_padded, eng_lengths, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val, len_train, len_val = train_test_split(
    X_train, y_train, len_train, test_size=0.1, random_state=42)
#%%

2025-07-26 22:05:47,779 INFO: Auto-selected CUDA (GPU)
2025-07-26 22:05:47,779 INFO: Auto-selected CUDA (GPU)
2025-07-26 22:05:47,779 INFO: Auto-selected CUDA (GPU)
INFO:__main__:Auto-selected CUDA (GPU)
2025-07-26 22:05:47,782 INFO: Final device: cuda
2025-07-26 22:05:47,782 INFO: Final device: cuda
2025-07-26 22:05:47,782 INFO: Final device: cuda
INFO:__main__:Final device: cuda


In [22]:
# 6. PyTorch Dataset and DataLoader
class TranslationDataset(Dataset):
    def __init__(self, src, tgt, src_lengths):
        self.src = torch.tensor(src, dtype=torch.long)
        self.tgt = torch.tensor(tgt, dtype=torch.long)
        self.src_lengths = torch.tensor(src_lengths, dtype=torch.long)
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx], self.src_lengths[idx]

train_ds = TranslationDataset(X_train, y_train, len_train)
val_ds = TranslationDataset(X_val, y_val, len_val)
test_ds = TranslationDataset(X_test, y_test, len_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

In [23]:
# 7. Encoder, Decoder, Seq2Seq with packed sequences
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_layers=3, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed_embedded = pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.rnn(packed_embedded)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_layers=3, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, input, hidden):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        hidden = self.encoder(src, src_lengths)
        input = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1
        return outputs


In [24]:
len(eng_vocab)

20275

In [6]:
def objective(trial):
    # Hyperparameter suggestions
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    tf_base = trial.suggest_float('tf_base', 0.3, 0.9)
    tf_decay = trial.suggest_float('tf_decay', 0.95, 1.0)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)  # Add dropout hyperparam


    encoder = Encoder(len(eng_vocab), emb_dim=128, hidden_dim=256, dropout=dropout).to(device)
    decoder = Decoder(len(nl_vocab), emb_dim=128, hidden_dim=256, dropout=dropout).to(device)
    model = Seq2Seq(encoder, decoder, device).to(device)

    optimizer = torch.optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss(ignore_index=0)


    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

    best_val_acc = 0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []

    for epoch in range(5):
        train_loss, train_acc = train(model, train_loader, optimizer, criterion, epoch,
                                      teacher_forcing_base=tf_base, teacher_forcing_decay=tf_decay)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        print(f"[Trial {trial.number}] Epoch {epoch+1} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        # Report for pruning
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Store final metrics
    trial.set_user_attr("final_train_loss", train_losses[-1])
    trial.set_user_attr("final_train_acc", train_accs[-1])
    trial.set_user_attr("final_val_loss", val_losses[-1])
    trial.set_user_attr("final_val_acc", val_accs[-1])
    trial.set_user_attr("all_train_losses", train_losses)
    trial.set_user_attr("all_train_accs", train_accs)
    trial.set_user_attr("all_val_losses", val_losses)
    trial.set_user_attr("all_val_accs", val_accs)

    return -best_val_acc  # Minimizing negative validation accuracy



In [18]:
def train(model, loader, optimizer, criterion, epoch, teacher_forcing_base=0.5, teacher_forcing_decay=0.99):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    # Compute decayed teacher forcing ratio
    teacher_forcing_ratio = max(teacher_forcing_base * (teacher_forcing_decay ** epoch), 0.1)

    for batch_idx, (src, tgt, src_lengths) in enumerate(tqdm(loader, desc=f"Training Epoch {epoch+1}")):
        src, tgt, src_lengths = src.to(device), tgt.to(device), src_lengths.to(device)

        optimizer.zero_grad()
        output = model(src, src_lengths, tgt, teacher_forcing_ratio=teacher_forcing_ratio)

        output_dim = output.shape[-1]
        output_flat = output[:, 1:].reshape(-1, output_dim)
        tgt_flat = tgt[:, 1:].reshape(-1)

        loss = criterion(output_flat, tgt_flat)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = output_flat.argmax(1)
        mask = tgt_flat != 0
        correct += (preds[mask] == tgt_flat[mask]).sum().item()
        total += mask.sum().item()

    accuracy = correct / total if total > 0 else 0
    return epoch_loss / len(loader), accuracy
def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for src, tgt, src_lengths in tqdm(loader, desc="Evaluating"):
            src, tgt, src_lengths = src.to(device), tgt.to(device), src_lengths.to(device)
            output = model(src, src_lengths, tgt, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output_flat = output[:, 1:].reshape(-1, output_dim)
            tgt_flat = tgt[:, 1:].reshape(-1)

            loss = criterion(output_flat, tgt_flat)
            epoch_loss += loss.item()

            preds = output_flat.argmax(1)
            mask = tgt_flat != 0
            correct += (preds[mask] == tgt_flat[mask]).sum().item()
            total += mask.sum().item()

    accuracy = correct / total if total > 0 else 0
    return epoch_loss / len(loader), accuracy



In [12]:
import optuna

# Create an Optuna study object
study = optuna.create_study(direction="minimize")  # since your objective returns negative val accuracy

# Run optimization for, say, 50 trials
study.optimize(objective, n_trials=50)

# Print best hyperparameters found
print("Best trial:")
trial = study.best_trial

print(f"  Value (negative val accuracy): {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


[I 2025-07-26 21:29:17,443] A new study created in memory with name: no-name-d6666273-a399-49ac-a9de-a9e0c6f143b8
Evaluating: 100%|██████████| 11/11 [00:00<00:00, 13.07it/s]


[Trial 0] Epoch 1 | Train Loss: 7.2018, Train Acc: 0.0661 | Val Loss: 6.0970, Val Acc: 0.0799


Evaluating: 100%|██████████| 11/11 [00:00<00:00, 13.62it/s]


[Trial 0] Epoch 2 | Train Loss: 6.0743, Train Acc: 0.0787 | Val Loss: 6.0507, Val Acc: 0.0815


Evaluating: 100%|██████████| 11/11 [00:00<00:00, 13.62it/s]


...

[Trial 18] Epoch 1 | Train Loss: 6.5268, Train Acc: 0.0770 | Val Loss: 6.0584, Val Acc: 0.0895


Evaluating: 100%|██████████| 11/11 [00:00<00:00, 13.54it/s]


[Trial 19] Epoch 1 | Train Loss: 7.0328, Train Acc: 0.0658 | Val Loss: 6.0631, Val Acc: 0.0804


Training Epoch 2:  65%|██████▍   | 59/91 [00:09<00:05,  6.29it/s]
[W 2025-07-26 21:43:49,524] Trial 19 failed with parameters: {'lr': 0.00018926835347741753, 'tf_base': 0.4411414911225392, 'tf_decay': 0.9511946047620651, 'dropout': 0.2144148181478024} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-6-2854920489.py", line 25, in objective
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, epoch,
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-11-1632154102.py", line 14, in train
    output = model(src, src_lengths, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/

KeyboardInterrupt: 

In [19]:
best_config = {
    "lr": 0.0006886,
    "tf_base": 0.3095,
    "tf_decay": 0.9839,
    "dropout": 0.2161
}

In [26]:
encoder = Encoder(len(eng_vocab), emb_dim=128, hidden_dim=256,dropout=best_config['dropout']).to(device)
decoder = Decoder(len(nl_vocab), emb_dim=128, hidden_dim=256).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)


optimizer = torch.optim.Adam(model.parameters(), lr=best_config['lr'])
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

best_val_acc = 0
train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(20):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, epoch,
                                  teacher_forcing_base=best_config['tf_base'], teacher_forcing_decay=best_config['tf_decay'])
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.48it/s]


Epoch 1 | Train Loss: 6.5585, Train Acc: 0.0987 | Val Loss: 6.3138, Val Acc: 0.1123


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.50it/s]


Epoch 2 | Train Loss: 6.1990, Train Acc: 0.1169 | Val Loss: 6.1637, Val Acc: 0.1181


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.46it/s]


...

Epoch 14 | Train Loss: 4.7877, Train Acc: 0.1812 | Val Loss: 5.8572, Val Acc: 0.1424


Evaluating: 100%|██████████| 61/61 [00:13<00:00,  4.40it/s]


Epoch 15 | Train Loss: 4.7168, Train Acc: 0.1840 | Val Loss: 5.8603, Val Acc: 0.1459


Training Epoch 16:  10%|█         | 56/546 [00:21<03:08,  2.60it/s]


KeyboardInterrupt: 

In [28]:
train_losses

[6.558513860562782,
 6.198987787023132,
 5.979615596624521,
 5.801022044031611,
 5.651448283003363,
 5.519500011052841,
 5.399661954942641,
 5.291979208097353,
 5.192802554085141,
 5.09539277387626,
 5.014217529541407,
 4.927979734791068,
 4.850291184889964,
 4.787701182313018,
 4.716824893986349]